# Batches on WEXAC

**Setup (once per clone):** `uv sync` on a compute node (`inode`), not the login node: it compiles the Rust engine.
Batch data goes to `$MORAN_GRAPHS_DATA` (default `/home/labs/pilpel/matanyaw/moran-graphs-data`), never into the repo.

**1. Write a plan** in the cell below and run it. It prints the graphs, run counts and a rough cost, and saves `plans/<batch>.json`.
A plan holds recipes (factory + kwargs + count), r values, repeats, seeds and LSF settings, never graphs.

**2. Submit** from a terminal in the repo:
```
uv run python -m moran_graphs.hpc submit plans/<batch>.json --dry-run   # builds every graph to check the plan, creates nothing (inode for large plans)
uv run python -m moran_graphs.hpc submit plans/<batch>.json             # light, fine on the login node: prepare job -> worker array -> finalize job
```
If prepare fails, LSF kills the workers and finalize; read `<batch>/logs/prepare_*.err`.

**3. Watch it:**
```
uv run python -m moran_graphs.hpc status <batch>     # chunks done / running / not started, LSF job states, rough ETA
bjobs -w                                             # raw LSF view
```

**4. If something failed** (finalize reports FAIL, or workers died):
```
uv run python -m moran_graphs.hpc resume <batch>     # frees claims of dead jobs, reruns only the missing chunks, then finalizes
uv run python -m moran_graphs.hpc finalize <batch>   # rebuild the report by hand (also checks the batch again)
```

**5. Read the report** in the cells at the end. Nothing in this notebook simulates; `load_report` raises if finalize has not run.
`<batch>/report/verification.txt` and `speed.txt` are also plain text you can `cat`.

**Testing without LSF** (on an inode): `submit plans/<batch>.json --local` runs the whole chain with local processes.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from moran_graphs.plan import describe, make_plan, save_plan, spec
from moran_graphs.hpc import DATA_ROOT, load_report
from moran_graphs.engine import WILD_TYPE

## Write a plan

In [ ]:
BATCH = "2026-10-22-small-test-batch"

specs = [
    spec("complete_graph", n_nodes=31),
    # spec("cycle_graph", n_nodes=31),
    # spec("star_graph", n_nodes=31, random_matches=10),
    # spec("grid_graph", width=5, height=6, random_matches=10),
    # spec("sample_random_graphs", n_nodes=31, n_edges=30, M=100, seed=1 )

    spec("mammalian_lung_graph", depth=4, skip_null_steps=True, random_matches=5),
    spec("avian_graph", n_parabronchi=7, l_parabronchi=4, random_matches=5),
    spec("fish_graph", n_rods=3, l_rod=3, fillaments=2, random_matches=5),
]

plan = make_plan(BATCH, specs, r_values=[1.1], n_repeats=1000,
                 zoo_seed=1, batch_seed=1, n_workers=100, queue="gsla-cpu")
describe(plan)
path = save_plan(plan, f"plans/{BATCH}.json")
print(f"submit with: \nuv run python -m moran_graphs.hpc submit {path}")
print(f"watch with:  watch -n 10 uv run python -m moran_graphs.hpc status {BATCH}")

In [ ]:
BATCH = "2026-10-08-classical-old-batch-31N-2"

specs = [
    spec("complete_graph", n_nodes=31),
    # spec("cycle_graph", n_nodes=31),
    # spec("star_graph", n_nodes=31, random_matches=10),
    # spec("grid_graph", width=5, height=6, random_matches=10),
    # spec("sample_random_graphs", n_nodes=31, n_edges=30, M=100, seed=1 )

    spec("mammalian_lung_graph", depth=4, skip_null_steps=True, random_matches=0),
    spec("avian_graph", n_parabronchi=7, l_parabronchi=4, random_matches=0),
    spec("fish_graph", n_rods=3, l_rod=3, fillaments=2, random_matches=0),
]
for n in range(30, 31):
    for e in range(-1, 5):
        specs.append(spec("random_graph", count=100, n_nodes=n, n_edges=n + e))



plan = make_plan(BATCH, specs, r_values=[1, 1.1, 1.2, 1.3, 2], n_repeats=100_000,
                 zoo_seed=1, batch_seed=1, n_workers=100, queue="gsla-cpu")
describe(plan)
path = save_plan(plan, f"plans/{BATCH}.json")
print(f"submit with: \nuv run python -m moran_graphs.hpc submit {path}")
print(f"watch with:  watch -n 10 uv run python -m moran_graphs.hpc status {BATCH}")

### Symmetric vs asymmetric lungs

flow_ratio 0.5 is the symmetric tree, 0.326 the human value. N only lands near 31 (28 to 33), and nearby flow ratios often give the same tree, which build_zoo drops as duplicates.

In [ ]:
BATCH = "2026-10-06-asymmetric-lungs"

# Rounded so names and plan JSON stay readable; 0.326 is not on the grid, so add it.
flow_ratios = sorted(set(np.round(np.linspace(0.01, 0.5, 50), 3)) | {0.326})
# N misses 31 in jumps, so each lung gets 5 random graphs of its own N and E.
lung_specs = [spec("asymmetric_lung_graph", flow_ratio=float(f), n_nodes=31, random_matches=5) for f in flow_ratios]
plan = make_plan(BATCH, lung_specs, r_values=[1.1], n_repeats=100_000,
                 zoo_seed=2, batch_seed=2, n_workers=100, queue="gsla-cpu",
                 description="asymmetric lungs, n_nodes=31, flow_ratio in (0, 0.5]")
describe(plan)
path = save_plan(plan, f"plans/{BATCH}.json")
print(f"submit with: uv run python -m moran_graphs.hpc submit {path}")
print(f"watch with:  watch -n 60 uv run python -m moran_graphs.hpc status {BATCH}")

In [ ]:
from moran_graphs import PopulationGraph as pg
import numpy as np
flow_ratios = sorted(set(np.round(np.linspace(0.01, 0.5, 10), 3)) | {0.326})
for f in flow_ratios:
    pg.asymmetric_lung_graph(flow_ratio=float(f), n_nodes=31).draw()

### Asymmetric lungs across sizes

flow_ratio 0.1 to 0.5 in steps of 0.05, plus 0.326 (human), at target N = 15, 31, ..., 1023: at 0.5 these are exactly the complete binary trees.
Other flow ratios land near the target (e.g. 964 to 1070 for 1023). 68 of 70 lung specs survive dedup, plus 10 random matches per (N, E): 568 graphs.

Cost: `describe` assumes N^2 steps per run, but lungs and random trees at N = 31 took about 9 N^2, so expect roughly 10x its figure (about 450 CPU h, under an hour on 1000 workers if the N^2 scaling holds; slower if large trees scale worse).

In [ ]:
BATCH = "2026-10-07-asymmetric-lungs-size-sweep"

HUMAN_FLOW_RATIO = 0.326  # Kundu & Panchagnula 2023
flow_ratios = sorted(set(np.round(np.arange(0.1, 0.5 + 1e-9, 0.05), 3)) | {HUMAN_FLOW_RATIO})
target_sizes = [2**k - 1 for k in range(4, 11)]  # 15 ... 1023, the sizes of complete binary trees
lung_specs = [spec("asymmetric_lung_graph", flow_ratio=float(f), n_nodes=n, random_matches=10)
              for n in target_sizes for f in flow_ratios]
plan = make_plan(BATCH, lung_specs, r_values=[1.1], n_repeats=100_000,
                 zoo_seed=3, batch_seed=3, n_workers=1000, queue="gsla-cpu",
                 description="asymmetric lungs, flow_ratio 0.1 to 0.5 by 0.05 and 0.326, n_nodes 15 to 1023")
describe(plan)
path = save_plan(plan, f"plans/{BATCH}.json")
print(f"check with:  uv run python -m moran_graphs.hpc submit {path} --dry-run")
print(f"submit with: uv run python -m moran_graphs.hpc submit {path}")
print(f"watch with:  watch -n 60 uv run python -m moran_graphs.hpc status {BATCH}")

### Respiratory graphs across sizes (retry of the v1 scaling batches)

Reruns `2026_08_31-scaling-{lung-2,avian,gills}` and `2026_09_02-scaling-more-shapes` in one batch, so every graph shares an engine and seeds.
Question: does fixation time keep the order fish > mammal > avian > complete as N grows from 15 to 1023?

- mammal: complete binary tree, depth 3 to 9, N = 15 ... 1023.
- avian: parabronchi of length 2, N = 2 n + 3 = 15 ... 1023.
- fish: filaments of 3 center nodes; 4 lamellae each at N = 16, then 10 lamellae (34 nodes per rod), N = 34 ... 1020.
- references: complete, grid, cycle, line, star at about the same N.
- every shape except complete and grid also runs directed (no random matches: random_graph is undirected).
- each undirected shape except complete gets 100 random graphs of its own N and E (shapes of the same N and E share them).


In [ ]:
BATCH = "2026-10-14-respiratory-size-sweep-2"

N_RANDOM = 100  # random graphs per (N, E)
SKIP = True  # Bd null-step skipping: same distribution, faster; random matches inherit it
# Directed graphs get no random matches: random_graph is undirected.
# Undirected stars run without skipping: every hub flip costs O(N) there, so skipping took
# ~2 days per chunk at N = 1023 in 2026-10-12-respiratory-size-sweep.
# Fish: one rod of length 3 with 10 lamellae is already 34 nodes, so N = 15 uses 4 lamellae.
specs = [
    # N = 15
    spec("complete_graph", n_nodes=15, skip_null_steps=SKIP),
    spec("grid_graph", width=4, height=4, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=15, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=15, directed=True, skip_null_steps=SKIP),
    spec("line_graph", n_nodes=15, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("line_graph", n_nodes=15, directed=True, skip_null_steps=SKIP),
    spec("star_graph", n_nodes=15, skip_null_steps=False, random_matches=N_RANDOM),
    spec("star_graph", n_nodes=15, directed=True, skip_null_steps=SKIP),
    spec("mammalian_lung_graph", depth=3, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("mammalian_lung_graph", depth=3, directed=True, skip_null_steps=SKIP),
    spec("avian_graph", n_parabronchi=6, l_parabronchi=2, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("avian_graph", n_parabronchi=6, l_parabronchi=2, directed=True, skip_null_steps=SKIP),
    spec("fish_graph", n_rods=1, l_rod=3, fillaments=4, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("fish_graph", n_rods=1, l_rod=3, fillaments=4, directed=True, skip_null_steps=SKIP),
    # N = 31
    spec("complete_graph", n_nodes=31, skip_null_steps=SKIP),
    spec("grid_graph", width=6, height=6, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=31, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=31, directed=True, skip_null_steps=SKIP),
    spec("line_graph", n_nodes=31, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("line_graph", n_nodes=31, directed=True, skip_null_steps=SKIP),
    spec("star_graph", n_nodes=31, skip_null_steps=False, random_matches=N_RANDOM),
    spec("star_graph", n_nodes=31, directed=True, skip_null_steps=SKIP),
    spec("mammalian_lung_graph", depth=4, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("mammalian_lung_graph", depth=4, directed=True, skip_null_steps=SKIP),
    spec("avian_graph", n_parabronchi=14, l_parabronchi=2, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("avian_graph", n_parabronchi=14, l_parabronchi=2, directed=True, skip_null_steps=SKIP),
    spec("fish_graph", n_rods=1, l_rod=3, fillaments=10, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("fish_graph", n_rods=1, l_rod=3, fillaments=10, directed=True, skip_null_steps=SKIP),
    # N = 63
    spec("complete_graph", n_nodes=63, skip_null_steps=SKIP),
    spec("grid_graph", width=8, height=8, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=63, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=63, directed=True, skip_null_steps=SKIP),
    spec("line_graph", n_nodes=63, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("line_graph", n_nodes=63, directed=True, skip_null_steps=SKIP),
    spec("star_graph", n_nodes=63, skip_null_steps=False, random_matches=N_RANDOM),
    spec("star_graph", n_nodes=63, directed=True, skip_null_steps=SKIP),
    spec("mammalian_lung_graph", depth=5, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("mammalian_lung_graph", depth=5, directed=True, skip_null_steps=SKIP),
    spec("avian_graph", n_parabronchi=30, l_parabronchi=2, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("avian_graph", n_parabronchi=30, l_parabronchi=2, directed=True, skip_null_steps=SKIP),
    spec("fish_graph", n_rods=2, l_rod=3, fillaments=10, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("fish_graph", n_rods=2, l_rod=3, fillaments=10, directed=True, skip_null_steps=SKIP),
    # N = 127
    spec("complete_graph", n_nodes=127, skip_null_steps=SKIP),
    spec("grid_graph", width=11, height=11, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=127, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=127, directed=True, skip_null_steps=SKIP),
    spec("line_graph", n_nodes=127, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("line_graph", n_nodes=127, directed=True, skip_null_steps=SKIP),
    spec("star_graph", n_nodes=127, skip_null_steps=False, random_matches=N_RANDOM),
    spec("star_graph", n_nodes=127, directed=True, skip_null_steps=SKIP),
    spec("mammalian_lung_graph", depth=6, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("mammalian_lung_graph", depth=6, directed=True, skip_null_steps=SKIP),
    spec("avian_graph", n_parabronchi=62, l_parabronchi=2, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("avian_graph", n_parabronchi=62, l_parabronchi=2, directed=True, skip_null_steps=SKIP),
    spec("fish_graph", n_rods=4, l_rod=3, fillaments=10, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("fish_graph", n_rods=4, l_rod=3, fillaments=10, directed=True, skip_null_steps=SKIP),
    # N = 255
    spec("complete_graph", n_nodes=255, skip_null_steps=SKIP),
    spec("grid_graph", width=16, height=16, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=255, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=255, directed=True, skip_null_steps=SKIP),
    spec("line_graph", n_nodes=255, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("line_graph", n_nodes=255, directed=True, skip_null_steps=SKIP),
    spec("star_graph", n_nodes=255, skip_null_steps=False, random_matches=N_RANDOM),
    spec("star_graph", n_nodes=255, directed=True, skip_null_steps=SKIP),
    spec("mammalian_lung_graph", depth=7, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("mammalian_lung_graph", depth=7, directed=True, skip_null_steps=SKIP),
    spec("avian_graph", n_parabronchi=126, l_parabronchi=2, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("avian_graph", n_parabronchi=126, l_parabronchi=2, directed=True, skip_null_steps=SKIP),
    spec("fish_graph", n_rods=8, l_rod=3, fillaments=10, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("fish_graph", n_rods=8, l_rod=3, fillaments=10, directed=True, skip_null_steps=SKIP),
    # N = 511
    spec("complete_graph", n_nodes=511, skip_null_steps=SKIP),
    spec("grid_graph", width=23, height=23, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=511, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=511, directed=True, skip_null_steps=SKIP),
    spec("line_graph", n_nodes=511, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("line_graph", n_nodes=511, directed=True, skip_null_steps=SKIP),
    spec("star_graph", n_nodes=511, skip_null_steps=False, random_matches=N_RANDOM),
    spec("star_graph", n_nodes=511, directed=True, skip_null_steps=SKIP),
    spec("mammalian_lung_graph", depth=8, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("mammalian_lung_graph", depth=8, directed=True, skip_null_steps=SKIP),
    spec("avian_graph", n_parabronchi=254, l_parabronchi=2, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("avian_graph", n_parabronchi=254, l_parabronchi=2, directed=True, skip_null_steps=SKIP),
    spec("fish_graph", n_rods=15, l_rod=3, fillaments=10, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("fish_graph", n_rods=15, l_rod=3, fillaments=10, directed=True, skip_null_steps=SKIP),
    # N = 1023
    spec("complete_graph", n_nodes=1023, skip_null_steps=SKIP),
    spec("grid_graph", width=32, height=32, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=1023, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("cycle_graph", n_nodes=1023, directed=True, skip_null_steps=SKIP),
    spec("line_graph", n_nodes=1023, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("line_graph", n_nodes=1023, directed=True, skip_null_steps=SKIP),
    spec("star_graph", n_nodes=1023, skip_null_steps=False, random_matches=N_RANDOM),
    spec("star_graph", n_nodes=1023, directed=True, skip_null_steps=SKIP),
    spec("mammalian_lung_graph", depth=9, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("mammalian_lung_graph", depth=9, directed=True, skip_null_steps=SKIP),
    spec("avian_graph", n_parabronchi=510, l_parabronchi=2, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("avian_graph", n_parabronchi=510, l_parabronchi=2, directed=True, skip_null_steps=SKIP),
    spec("fish_graph", n_rods=30, l_rod=3, fillaments=10, skip_null_steps=SKIP, random_matches=N_RANDOM),
    spec("fish_graph", n_rods=30, l_rod=3, fillaments=10, directed=True, skip_null_steps=SKIP),
]

plan = make_plan(BATCH, specs, r_values=[1.1], n_repeats=100_000,
                 zoo_seed=4, batch_seed=4, n_workers=950, queue="gsla-cpu",
                 description="complete, grid, cycle, line, star, mammal, avian, fish (+ directed) and random matches, N 15 to 1023")
describe(plan)
path = save_plan(plan, f"plans/{BATCH}.json")
print(f"check with:  uv run python -m moran_graphs.hpc submit {path} --dry-run")
print(f"submit with: uv run python -m moran_graphs.hpc submit {path}")
print(f"watch with:  watch -n 10 uv run python -m moran_graphs.hpc status {BATCH}")

## Read a finished batch

In [ ]:
rep = load_report(BATCH)
print(rep["verification"]["overall"])
for c in rep["verification"]["checks"]:
    print(f"[{c['status']:<4}] {c['check']:<15} {c['message']}")
results = rep["results"]
results[["name", "r", "n", "rho", "rho_lo", "rho_hi", "steps_mean_given_fixed", "n_censored"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for r, part in results.groupby("r"):
    part = part.sort_values("name")
    ax.errorbar(part["name"], part["rho"], yerr=[part["rho"] - part["rho_lo"], part["rho_hi"] - part["rho"]],
                fmt="o", capsize=3, label=f"r = {r}")
ax.set(ylabel="fixation probability (95% Wilson CI)")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
plt.tight_layout()

## How the batch ran

In [ ]:
for key, value in rep["speed"].items():
    print(f"{key:28s} {value}")
t = rep["chunk_timing"]
fig, (ax_w, ax_d) = plt.subplots(1, 2, figsize=(13, 4))
t0 = t["start"].min()
for k, (worker, part) in enumerate(t.groupby(["job", "index"])):
    ax_w.barh(k, part["end"] - part["start"], left=part["start"] - t0, height=0.8, color="#2a78d6")
ax_w.set(xlabel="seconds since the first chunk started", ylabel="worker", title="Chunks per worker (gaps are idle time)")
ax_d.hist(t["end"] - t["start"], bins=30, color="#2a78d6")
ax_d.set(xlabel="chunk wall time (s)", ylabel="chunks", title="Chunk durations (tune chunk_target_steps)")
plt.tight_layout()